### Imports and R3 data

In [14]:
import random
import itertools

# -----------------------------
# R3 data from Table 1
# -----------------------------
# Job : [Op1, Op2]
TIME = {
    1: [3, 6],
    2: [10, 1],
    3: [3, 2],
    4: [2, 4],
    5: [8, 8]
}

JOBS = list(TIME.keys())
M = 2   # number of machines
N = 2   # operations per job

### Makespan and fitness

In [15]:
# -----------------------------
# Makespan / fitness from R2
# -----------------------------
def compute_makespan(chromosome, time_table, M, N):
    machine_free = [0] * M
    job_ready = {job: 0 for job in time_table}
    makespan = 0

    for job in chromosome:
        for k in range(N):
            machine = k % M
            start = max(machine_free[machine], job_ready[job])
            finish = start + time_table[job][k]
            machine_free[machine] = finish
            job_ready[job] = finish
            makespan = max(makespan, finish)

    return makespan


def compute_fitness(chromosome, time_table, M, N):
    ms = compute_makespan(chromosome, time_table, M, N)
    return 1 / (1 + ms)

### Crossover

In [16]:
# -----------------------------
# Two-cut crossover from R1
# -----------------------------
def crossover(p1, p2):
    J = len(p1)
    cut1, cut2 = sorted(random.sample(range(J), 2))

    def build_child(keep_parent, fill_parent):
        child = [None] * J

        # copy middle segment
        child[cut1:cut2+1] = keep_parent[cut1:cut2+1]
        used = set(child[cut1:cut2+1])

        # remaining genes from fill_parent in same relative order
        fill_values = [x for x in fill_parent if x not in used]

        pos = 0
        for i in range(J):
            if child[i] is None:
                child[i] = fill_values[pos]
                pos += 1

        return child

    child1 = build_child(p1, p2)
    child2 = build_child(p2, p1)
    return child1, child2

### Mutation

In [17]:
# -----------------------------
# Mutation: swap two positions
# -----------------------------
def mutate(chromosome, prob=0.05):
    c = chromosome[:]
    if random.random() < prob:
        i, j = random.sample(range(len(c)), 2)
        c[i], c[j] = c[j], c[i]
    return c

### Initial Population

In [18]:
# -----------------------------
# Initial population
# -----------------------------
def generate_initial_population(size=120):
    population = []
    seen = set()

    while len(population) < size:
        perm = tuple(random.sample(JOBS, len(JOBS)))
        if perm not in seen:
            seen.add(perm)
            population.append(list(perm))

    return population

### Selection

In [19]:
# -----------------------------
# Selection: choose 10 according to fitness probability
# -----------------------------
def select_population(population, k=10):
    fitnesses = [compute_fitness(ch, TIME, M, N) for ch in population]
    total_fit = sum(fitnesses)
    probs = [f / total_fit for f in fitnesses]
    selected = random.choices(population, weights=probs, k=k)
    return [ch[:] for ch in selected]

### GA main function 

In [20]:
# -----------------------------
# GA for R3
# -----------------------------
def run_ga_r3(generations=100, seed=42):
    random.seed(seed)

    population = generate_initial_population(120)

    best_chromosome = None
    best_makespan = float('inf')

    for gen in range(generations):
        # update best from current population
        for ch in population:
            ms = compute_makespan(ch, TIME, M, N)
            if ms < best_makespan:
                best_makespan = ms
                best_chromosome = ch[:]

        # select 10 chromosomes
        selected = select_population(population, k=10)

        # crossover for all pairs among the 10 selected
        new_children = []
        for i in range(len(selected)):
            for j in range(i + 1, len(selected)):
                c1, c2 = crossover(selected[i], selected[j])
                new_children.append(mutate(c1, 0.05))
                new_children.append(mutate(c2, 0.05))

        # add children to population
        population.extend(new_children)

    return best_chromosome, best_makespan

### Run and test

In [21]:
# -----------------------------
# Run R3
# -----------------------------
best_chromosome, best_makespan = run_ga_r3()

print("Best chromosome found:", best_chromosome)
print("Best makespan:", best_makespan)

# verify the worked example
example = [4, 1, 5, 3, 2]
print("Makespan of [4,1,5,3,2]:", compute_makespan(example, TIME, M, N))

Best chromosome found: [4, 1, 3, 5, 2]
Best makespan: 27
Makespan of [4,1,5,3,2]: 27
